<a href="https://colab.research.google.com/github/emilyrgarman/ml3finalproject_triage/blob/main/model_building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
! git clone https://github.com/emilyrgarman/ml3finalproject_triage.git

fatal: destination path 'ml3finalproject_triage' already exists and is not an empty directory.


In [34]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

SEED = 42

Loading in and merging train and test sets with chief complaints text data

In [35]:
train = pd.read_csv('/content/ml3finalproject_triage/train.csv')
test = pd.read_csv('/content/ml3finalproject_triage/test.csv')
complaints = pd.read_csv('/content/ml3finalproject_triage/chief_complaints.csv')

In [36]:
train_df = train.merge(complaints[['patient_id', 'chief_complaint_raw']], on='patient_id', how='left')
test_df  = test.merge(complaints[['patient_id', 'chief_complaint_raw']], on='patient_id', how='left')

In [37]:
train_df.head()

,patient_id,site_id,triage_nurse_id,arrival_mode,arrival_hour,arrival_day,arrival_month,arrival_season,shift,age,...,pain_score,weight_kg,height_cm,bmi,shock_index,news2_score,disposition,ed_los_hours,triage_acuity,chief_complaint_raw
0,TG-UXRGA9UCO,SITE-TMP-01,NURSE-0033,walk-in,6,Monday,5,spring,morning,43,...,7,52.3,165.4,19.1,0.725,8,discharged,7.35,2,"thunderclap headache, worsening with movement"
1,TG-B19DBBS2G,SITE-HEL-01,NURSE-0001,walk-in,6,Thursday,4,spring,morning,72,...,-1,73.3,164.4,27.1,0.739,1,discharged,0.70,5,"contraception advice, intermittent"
2,TG-GZ97W7M6V,SITE-HEL-02,NURSE-0005,walk-in,8,Saturday,4,spring,morning,82,...,3,77.1,183.7,22.8,0.798,2,discharged,0.63,5,"general health question, intermittent"
3,TG-THIB2TN9Q,SITE-HEL-02,NURSE-0026,police,7,Sunday,3,spring,morning,50,...,7,49.6,172.6,16.6,0.812,2,discharged,1.99,3,"erythema migrans tick bite, intermittent"
4,TG-J3U3LQ2QY,SITE-HEL-02,NURSE-0044,walk-in,5,Tuesday,5,spring,night,62,...,4,71.9,173.4,23.9,0.812,2,transferred,3.58,3,"cellulitis localised, intermittent"


In [38]:
train_df.columns

Index(['patient_id', 'site_id', 'triage_nurse_id', 'arrival_mode',
       'arrival_hour', 'arrival_day', 'arrival_month', 'arrival_season',
       'shift', 'age', 'age_group', 'sex', 'language', 'insurance_type',
       'transport_origin', 'pain_location', 'mental_status_triage',
       'chief_complaint_system', 'num_prior_ed_visits_12m',
       'num_prior_admissions_12m', 'num_active_medications',
       'num_comorbidities', 'systolic_bp', 'diastolic_bp',
       'mean_arterial_pressure', 'pulse_pressure', 'heart_rate',
       'respiratory_rate', 'temperature_c', 'spo2', 'gcs_total', 'pain_score',
       'weight_kg', 'height_cm', 'bmi', 'shock_index', 'news2_score',
       'disposition', 'ed_los_hours', 'triage_acuity', 'chief_complaint_raw'],
      dtype='object')

In [41]:
len(train_df)

80000

Preprocessing and train/validation split

In [39]:
## need to update these im confused
NUMERIC_COLS = ['age', 'num_prior_ed_visits_12m', 'num_prior_admissions_12m',
                'num_active_medications', 'num_comorbidities',
                'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure',
                'pulse_pressure', 'heart_rate',
                'respiratory_rate', 'temperature_c', 'spo2',
                'gcs_total', 'pain_score', 'weight_kg',
                'height_cm', 'bmi', 'shock_index', 'news2_score',
                'ed_los_hours']
CATEG_COLS   = ['arrival_mode', 'arrival_hour', 'arrival_day',
                'arrival_month', 'arrival_season', 'shift',
                'age_group', 'sex', 'language', 'insurance_type',
                'transport_origin', 'pain_location', 'mental_status_triage',
                'disposition', 'chief_complaint_system']
ID_COLS      = ['patient_id', 'site_id', 'triage_nurse_id']
TARGET_COL   = ['triage_acuity']
TEXT_COL     = ['chief_complaint_raw']
TABULAR_COLS = NUMERIC_COLS + CATEG_COLS

for col in CATEG_COLS:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    test_df[col]  = le.transform(test_df[col].astype(str))

imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_tab      = scaler.fit_transform(imputer.fit_transform(train_df[TABULAR_COLS]))
X_tab_test = scaler.transform(imputer.transform(test_df[TABULAR_COLS]))

# Shift labels 1–5 → 0–4 for CrossEntropyLoss
y = train_df[TARGET_COL].values - 1

# ── 4. Train/val split ─────────────────────────────────────────────
X_tr, X_val, y_tr, y_val, idx_tr, idx_val = train_test_split(
    X_tab, y, np.arange(len(y)),
    test_size=0.3, stratify=y, random_state=SEED
)

texts_all  = train_df[TEXT_COL].fillna('').values
texts_tr   = texts_all[idx_tr]
texts_val  = texts_all[idx_val]
texts_test = test_df[TEXT_COL].fillna('').values

print(f'\nTrain: {len(y_tr)} | Val: {len(y_val)} | Test: {len(test_df)}')

KeyError: 'disposition'